> **Pipeline status:** Optional historical LPPLS fitting notebook. Do not run it when complete, validated LPPLS CSV files have already been supplied. Run scripts/audit_inputs.py instead.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import random

import numpy as np
import pandas as pd
import yfinance as yf
from lppls import lppls
from tqdm.auto import tqdm

# Parametry oryginalnego eksperymentu
TICKER = "BTC-USD"
START = "2015-01-01"
END_EXCLUSIVE = "2025-01-01"  # dzień 31.12.2024 będzie uwzględniony
MIN_WIN = 30
MAX_WIN = 180
STEP = 5
MAX_SEARCHES = 50
SEED = 42

OUTPUT_FILE = Path("lppls_rolling_all_fits_2015_2024.csv")
BATCH_SIZE = 200  # tylko częstotliwość dopisywania do tego samego CSV

COLUMNS = [
    "window_id", "run_id", "win_start", "win_end", "n", "ok", "err",
    "tc", "m", "w", "A", "B", "C", "c1", "c2", "O", "D",
    "ssr", "r2", "t_gap"
]


In [ ]:
# 1) Dane
data = yf.download("BTC-USD", start="2015-01-01", end="2025-01-01")
# data = yf.download("^GSPC", start="2000-01-01", end="2025-08-31")

data = data.reset_index()

price_col = "Adj Close" if "Adj Close" in data.columns else "Close"
df = data[["Date", price_col]].dropna().sort_values("Date").reset_index(drop=True)
df = df.rename(columns={price_col: "price"})

# 2) Wektory czasu i ceny (1D)
time = df["Date"].map(pd.Timestamp.toordinal).to_numpy(dtype=float)
price = np.log(df["price"].to_numpy(dtype=float))

# 3) Jednolita filtracja niefinitych wierszy (unikamy problemów z maską)
stack = np.column_stack([time, price])             # shape (N, 2)
stack = stack[np.isfinite(stack).all(axis=1)]      # tylko wiersze w pełni „finite”
time, price = stack[:, 0], stack[:, 1]             # z powrotem na 2 wektory 1D

# 4) Observations: (2, N)
observations = np.vstack([time, price])

# 5) LPPLS
MAX_SEARCHES = 50
model = lppls.LPPLS(observations=observations)
tc, m, w, a, b, c, c1, c2, O, D = model.fit(MAX_SEARCHES)
model.plot_fit(show_tc=True)
print(tc, m, w, a, b, c, c1, c2, O, D)

In [ ]:
df = df.copy() 
df["t_ord"] = df["Date"].map(pd.Timestamp.toordinal).astype(float) 
df["logp"] = np.log(df["price"].astype(float)) 

# Ustawienia okien 
MIN_WIN = 30
MAX_WIN = 180
STEP = 5 
MAX_SEARCHES = 50 
records = [] 
def fit_lppls_on_window(t, p, max_searches=50): 
    obs = np.vstack([t, p]) 
    model = lppls.LPPLS(observations=obs) 
    try: 
        tc, m, w, A, B, C, c1, c2, O, D = model.fit(max_searches) 
    except Exception as e: 
        return {"ok": False, "err": str(e)} 
    
    p_hat = model.lppls(t, tc, m, w, A, B, c1, c2) 
    resid = p - p_hat 
    ssr = float(np.sum(resid**2)) 
    sst = float(np.sum((p - p.mean())**2)) 
    r2 = float(1.0 - ssr / sst) if sst > 0 else np.nan 
    # t_start = float(t[0])
    t_end = float(t[-1]) 
    
    ok_m = (0 < m < 1) 
    # ok_m = (0 < m < 1.5) 
    ok_w = (6 <= w <= 13) 
    # ok_w = (4 <= w <= 16) 
    ok_B = (B < 0) 
    # ok_tc = (tc > t_end)
    ok_tc = (0 < (tc - t_end) <= 40)
    # ok_tc = (tc > t_start) & (tc <= t_end + 30)
    ok_r2 = (r2 > 0.9) 
 
    
    return {
        "ok": True,
        "tc": float(tc), "m": float(m), "w": float(w),
        "A": float(A), "B": float(B), "C": float(C),
        "c1": float(c1), "c2": float(c2),
        "O": float(O), "D": float(D),
        "ssr": ssr, "r2": r2,
        "cond_m": ok_m, "cond_w": ok_w, "cond_B": ok_B, "cond_tc": ok_tc
    }
 # Pętla po oknach: różne długości i przesunięcia 
N = len(df)
for start in tqdm(range(0, N - MIN_WIN + 1, STEP), desc="Rolling windows"):
    for L in range(MIN_WIN, min(MAX_WIN, N - start) + 1):
        end = start + L
        t_win = df["t_ord"].to_numpy()[start:end]
        p_win = df["logp"].to_numpy()[start:end]
        res = fit_lppls_on_window(t_win, p_win)

        row = {
            "win_start": df.loc[start, "Date"],
            "win_end":   df.loc[end-1, "Date"],
            "n": L,
        }
        row.update(res)
        records.append(row)

rolling_results = pd.DataFrame.from_records(records)


In [ ]:
rolling_results.to_csv("lppls_rolling_results.csv", index=False)

In [ ]:
rolling_results